<a href="https://colab.research.google.com/github/240264-sudo/AI-ML-lab/blob/main/AI_ML_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Artificial Neural Network (ANN) for Classification

import numpy as np
import tensorflow as tf
from sklearn.datasets import load_irisfrom sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1: Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Step 2: Split the dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 3: Normalize the input features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 4: Create the Artificial Neural Network
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

# Step 5: Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Step 6: Train the model
model.fit(X_train, y_train, epochs=100, batch_size=8, verbose=0)

# Step 7: Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("Test Loss:", round(loss, 4))
print("Test Accuracy:", round(accuracy * 100, 2), "%")

# Step 8: Make predictions
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Step 9: Display the confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Step 10: Display classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=iris.target_names,
    zero_division=0
))

# Step 11: Predict the species of a new flower
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])
new_flower = scaler.transform(new_flower)

prediction = model.predict(new_flower, verbose=0)
predicted_class = np.argmax(prediction, axis=1)[0]

print("Predicted Flower Species:", iris.target_names[predicted_class])

Test Loss: 0.0682
Test Accuracy: 100.0 %

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Predicted Flower Species: setosa


In [ ]:
# Artificial Neural Network (ANN) - Online Food Ordering Dataset

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from pathlib import Path

# Step 1: Load the dataset (excluding standard sample data)
csv_files = [f for f in Path(".").rglob("*.csv") if "sample_data" not in str(f)]

if csv_files:
    csv_file = csv_files[0]
    df = pd.read_csv(csv_file)
    print("Dataset loaded:", csv_file.name)
else:
    print("No custom CSV file found. Generating synthetic Online Food Ordering dataset for demonstration...")
    # Generate a synthetic dataset mimicking the online food ordering structure
    np.random.seed(42)
    n_samples = 200
    df = pd.DataFrame({
        'Age': np.random.randint(18, 50, size=n_samples),
        'Gender': np.random.choice(['Male', 'Female'], size=n_samples),
        'Marital Status': np.random.choice(['Single', 'Married'], size=n_samples),
        'Occupation': np.random.choice(['Student', 'Employee', 'Self Employed'], size=n_samples),
        'Monthly Income': np.random.choice(['No Income', 'Below 10000', '10000 to 25000', 'More than 50000'], size=n_samples),
        'Educational Qualifications': np.random.choice(['Graduate', 'Post Graduate', 'Under Graduate'], size=n_samples),
        'Family size': np.random.randint(1, 6, size=n_samples),
        'Feedback': np.random.choice(['Positive', 'Negative'], size=n_samples),
        'Output': np.random.choice(['Yes', 'No'], size=n_samples)
    })

print("\nFirst five rows:")
print(df.head())

print("\nDataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# Step 2: Clean column names
df.columns = df.columns.astype(str).str.strip()

# Identify the target column
target_column = next(
    (col for col in df.columns if col.lower() == "output"),
    None
)

if target_column is None:
    raise ValueError(
        "Target column 'Output' was not found. "
        "Check the column names printed above."
    )

# Remove rows without a target value
df = df.dropna(subset=[target_column]).copy()
df[target_column] = df[target_column].astype(str).str.strip()
df = df[df[target_column] != ""]

# Step 3: Separate features and target
X = df.drop(columns=[target_column])

# Remove Feedback to avoid using potentially outcome-related information
X = X.drop(
    columns=[
        col for col in X.columns
        if col.lower() == "feedback"
        or col.lower().startswith("unnamed")
    ],
    errors="ignore"
)

y = df[target_column]

# Encode target labels into numbers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print("\nTarget classes:", label_encoder.classes_)
print("Number of features before encoding:", X.shape[1])

# Step 4: Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Step 5: Identify numerical and categorical columns
numerical_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Handle missing numerical values and scale the data
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Handle missing categorical values and encode categories
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder)
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

X_train = np.asarray(X_train, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# Step 6: Create the ANN model
num_classes = len(label_encoder.classes_)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu")
])

# Configure the output layer for classification
if num_classes == 2:
    model.add(tf.keras.layers.Dense(1, activation="sigmoid"))
    loss_function = "binary_crossentropy"
else:
    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))
    loss_function = "sparse_categorical_crossentropy"

# Step 7: Compile the model
model.compile(
    optimizer="adam",
    loss=loss_function,
    metrics=["accuracy"]
)

# Step 8: Train the ANN
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=0
)

print("\nANN training completed.")
print("Epochs trained:", len(history.history["loss"]))

# Step 9: Evaluate the model
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Loss:", round(loss, 4))
print("Test Accuracy:", round(accuracy * 100, 2), "%")

# Step 10: Generate predictions
predictions = model.predict(X_test, verbose=0)

if num_classes == 2:
    y_pred = (predictions.ravel() >= 0.5).astype(int)
else:
    y_pred = np.argmax(predictions, axis=1)

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(num_classes)
))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(num_classes),
    target_names=label_encoder.classes_,
    zero_division=0
))

# Step 11: Display an example prediction
actual = label_encoder.inverse_transform([int(y_test[0])])[0]
predicted = label_encoder.inverse_transform([int(y_pred[0])])[0]

print("\nExample Prediction:")
print("Actual Output:", actual)
print("Predicted Output:", predicted)

No custom CSV file found. Generating synthetic Online Food Ordering dataset for demonstration...

First five rows:
   Age  Gender Marital Status     Occupation   Monthly Income  \
0   24    Male        Married        Student      Below 10000   
1   37  Female        Married       Employee        No Income   
2   46    Male         Single  Self Employed  More than 50000   
3   32    Male         Single  Self Employed      Below 10000   
4   28  Female        Married  Self Employed        No Income   

  Educational Qualifications  Family size  Feedback Output  
0                   Graduate            3  Positive    Yes  
1                   Graduate            5  Negative     No  
2              Post Graduate            5  Positive     No  
3              Post Graduate            1  Negative    Yes  
4             Under Graduate            1  Positive    Yes  

Dataset shape: (200, 9)

Columns: ['Age', 'Gender', 'Marital Status', 'Occupation', 'Monthly Income', 'Educational Qualificatio


Test Loss: 0.7318
Test Accuracy: 55.0 %

Confusion Matrix:
[[ 0 18]
 [ 0 22]]

Classification Report:
              precision    recall  f1-score   support

          No       0.00      0.00      0.00        18
         Yes       0.55      1.00      0.71        22

    accuracy                           0.55        40
   macro avg       0.28      0.50      0.35        40
weighted avg       0.30      0.55      0.39        40


Example Prediction:
Actual Output: No
Predicted Output: Yes
